# Stepageddon - Train Step Chart Model

Train a neural network to generate DDR step charts from audio.

**Prerequisites:**
1. Run `prepare_data.py` locally to preprocess your charts
2. Zip the training data: `cd backend/ml && zip -r training_data.zip training_data/`
3. Upload `training_data.zip` directly to Colab when prompted (cell below)

**Runtime:** Select GPU (T4) under Runtime > Change runtime type

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q librosa numpy torch simfile

In [ ]:
# Upload the ml/ module code
from google.colab import files
import os

os.makedirs('/content/ml', exist_ok=True)
# Make /content/ml a real Python package so `import ml.train` works
open('/content/ml/__init__.py', 'w').close()

# Upload all four: model.py, dataset.py, train.py, prepare_data.py
# (prepare_data is imported by dataset.py for the DB_RANGE / FORMAT_VERSION constants)
print('Upload: model.py, dataset.py, train.py, AND prepare_data.py (from backend/ml/)')
uploaded = files.upload()
for name, data in uploaded.items():
    out_path = os.path.join('/content/ml', os.path.basename(name))
    with open(out_path, 'wb') as f:
        f.write(data)
    print(f'Wrote {out_path}')

!ls -la /content/ml/


In [ ]:
# Upload and extract training data
# Pick ONE option below and comment out the others

# --- Option A: Copy from Google Drive to local disk (most reliable for large files) ---
# Upload training_data.zip to your Google Drive first (any location), then:
import zipfile, os
ZIP_PATH = '/content/drive/MyDrive/stepageddon/training_data.zip'  # adjust path if needed
print(f'Copying and extracting from Drive...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
print('Done!')

# --- Option B: Direct URL download (if hosted somewhere) ---
# !wget -q -O /content/training_data.zip "YOUR_URL_HERE"
# import zipfile
# with zipfile.ZipFile('/content/training_data.zip', 'r') as z:
#     z.extractall('/content/')
# !rm /content/training_data.zip

# --- Option C: Browser upload (only works for small files <1GB) ---
# from google.colab import files
# uploaded = files.upload()
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
#     z.extractall('/content/')

!ls /content/training_data/ | head -20

In [ ]:
# Verify data is accessible (manifest format_version=2)
import json
from pathlib import Path

DATA_DIR = Path('/content/training_data')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/stepageddon/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

assert manifest.get('format_version') == 2, (
    f"Expected manifest format_version=2, got {manifest.get('format_version')}. "
    "Re-run prepare_data.py."
)
entries = manifest['entries']
print(f'Total training examples: {len(entries)}')
print(f'Mel whitening stats: mean={manifest["mel_mean"]:.4f} std={manifest["mel_std"]:.4f}')

# Show difficulty distribution
from collections import Counter
diff_counts = Counter(e['difficulty'] for e in entries)
print(f'Difficulty distribution: {dict(diff_counts)}')

# Check a sample file (one npz per song; labels live under labels_<diff>)
import numpy as np
sample_entry = entries[0]
sample = np.load(DATA_DIR / sample_entry['filename'])
labels = sample[sample_entry['labels_key']]
print(f"\nSample: {sample_entry['song_title']} ({sample_entry['difficulty']})")
print(f'  Mel shape: {sample["mel"].shape}')
print(f'  Labels shape: {labels.shape}')
print(f'  Note frames: {(labels > 0).any(axis=1).sum()}')


In [ ]:
# Add to Python path
import sys
sys.path.insert(0, '/content')

# Also need the modules for schema imports during inference (not needed for training)
# For training only, we just need ml.model and ml.dataset

In [ ]:
# Build dataloaders, model, optimizer, loss using train.py helpers.
# Keep the Colab-specific config here; delegate all training logic to ml.train
# so the notebook and the CLI always run the exact same code path.
from types import SimpleNamespace
import torch
from torch.amp import GradScaler
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

from ml.train import (
    build_dataloaders, build_model, build_optimizer_and_scheduler,
    build_loss, seed_everything,
)

# Hyperparameters
args = SimpleNamespace(
    data_dir=str(DATA_DIR),
    checkpoint_dir=str(CHECKPOINT_DIR),
    epochs=80,
    batch_size=32,
    lr=3e-4,
    hidden_dim=256,
    n_heads=8,
    n_layers=4,
    chunk_frames=500,        # 5s at ~100fps
    val_split=0.1,
    weight_decay=0.01,
    warmup_epochs=2.0,
    type_weight=1.0,
    duration_weight=1.0,
    # Type-head class rebalancing (new). 0.5 sqrt-smooths inverse frequency,
    # capped at 12x. Lower smoothing = more aggressive rare-class upweighting
    # but more unstable; raise to 0.7+ if jump/hold loss spikes destabilize.
    type_weight_smoothing=0.5,
    type_weight_cap=12.0,
    dropout=0.1,
    num_workers=2,
    ema_decay=0.999,
    seed=42,
    tol_frames=3,            # ~30ms at 100fps
)

seed_everything(args.seed)
device = torch.device('cuda')

built = build_dataloaders(args)
# build_model and build_loss now take type_prior so the type head's bias is
# initialized to log(prior) and CE is class-weighted.
model = build_model(args, built.onset_prior, built.type_prior, device)
criterion = build_loss(built.onset_prior, built.type_prior, args).to(device)
optimizer, scheduler = build_optimizer_and_scheduler(
    model, args, steps_per_epoch=max(1, len(built.train_loader)),
)
scaler = GradScaler('cuda')
ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(args.ema_decay))

print(f'Train entries: {len(built.train_idx)}  Val entries: {len(built.val_idx)}')
print(f'Empirical onset prior: {built.onset_prior:.6f}')
print(f'Empirical type prior (tap/jump/hold): {built.type_prior.tolist()}')
print(f'Empirical density/difficulty: {built.default_density_by_id.tolist()}')


In [ ]:
# Training loop — calls into ml.train.train_one_epoch / validate.
# Auto-resumes from last_model.pt on Drive if it exists, so re-running this
# cell after a Colab disconnect picks up where it left off instead of
# restarting from epoch 0.
import time
from ml.train import train_one_epoch, validate, save_checkpoint

history = {'train_loss': [], 'val_loss': [], 'tol_f1': [], 'macro_f1': []}
best_f1 = -1.0
start_epoch = 0

# Resume from last checkpoint if present (survives Colab disconnects).
last_ckpt_path = CHECKPOINT_DIR / 'last_model.pt'
if last_ckpt_path.exists():
    print(f'Resuming from {last_ckpt_path}')
    ckpt = torch.load(last_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    if ckpt.get('ema_state_dict') is not None:
        ema_model.module.load_state_dict(ckpt['ema_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_f1 = ckpt.get('metrics', {}).get('tol_f1', -1.0)
    print(f'  -> resumed at epoch {start_epoch}, best tol_f1={best_f1:.3f}')
else:
    print('No checkpoint found — starting fresh from epoch 0.')

for epoch in range(start_epoch, args.epochs):
    t0 = time.time()
    train_metrics = train_one_epoch(
        model, built.train_loader, criterion, optimizer, scheduler,
        scaler, device, ema_model=ema_model,
    )
    val_metrics = validate(
        ema_model.module, built.val_loader, criterion, device,
        tol_frames=args.tol_frames,
    )

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['tol_f1'].append(val_metrics['tol_f1'])
    history['macro_f1'].append(val_metrics['type_macro_f1'])

    # Per-class F1 in the print-out is the new signal for whether the model
    # is actually learning to predict jumps/holds — type_acc alone hid the
    # "always tap" failure mode.
    print(
        f"Epoch {epoch+1:3d}/{args.epochs} | "
        f"train={train_metrics['loss']:.4f} "
        f"(onset={train_metrics['onset_loss']:.4f} type={train_metrics['type_loss']:.4f}) "
        f"| val={val_metrics['loss']:.4f} "
        f"tol_f1={val_metrics['tol_f1']:.3f} "
        f"(P={val_metrics['tol_precision']:.3f} R={val_metrics['tol_recall']:.3f}) "
        f"type_F1[t/j/h]={val_metrics['tap_f1']:.3f}/"
        f"{val_metrics['jump_f1']:.3f}/{val_metrics['hold_f1']:.3f} "
        f"(macro={val_metrics['type_macro_f1']:.3f}) "
        f"dur_mae={val_metrics['dur_mae']:.3f}s "
        f"| lr={lr:.2e} | {elapsed:.1f}s",
        flush=True,
    )

    # save_checkpoint now also persists type_prior so inference can apply
    # logit adjustment without a heuristic fallback.
    save_checkpoint(
        CHECKPOINT_DIR / 'last_model.pt', epoch, model, ema_model,
        optimizer, scheduler, scaler, val_metrics,
        built.default_density_by_id,
        built.full_dataset.mel_mean, built.full_dataset.mel_std,
        built.type_prior,
        args,
    )
    if val_metrics['tol_f1'] > best_f1:
        best_f1 = val_metrics['tol_f1']
        save_checkpoint(
            CHECKPOINT_DIR / 'best_model.pt', epoch, model, ema_model,
            optimizer, scheduler, scaler, val_metrics,
            built.default_density_by_id,
            built.full_dataset.mel_mean, built.full_dataset.mel_std,
            built.type_prior,
            args,
        )
        print(f'  -> New best (tol_f1={best_f1:.3f})', flush=True)

print(f'\nTraining complete! Best tol_f1={best_f1:.3f}')
print(f'Best model saved at: {CHECKPOINT_DIR}/best_model.pt')


## After Training

1. Download `best_model.pt` from Google Drive (`stepageddon/checkpoints/best_model.pt`)
2. Place it in `backend/ml/checkpoints/best_model.pt`
3. Set `USE_ML_GENERATION=true` in `backend/.env`
4. Restart the backend server